# Toffoli (CCX)

The Toffoli gate has two controls and one target. It flips the target if
and only if **both** controls are $|1\rangle$:

$$
\mathrm{CCX}|c_1 c_0 t\rangle = |c_1 c_0\ (t \oplus c_1 c_0)\rangle.
$$

With $t=0$ this is a reversible AND. The notebook is standalone — it
does not import the other `basic/` examples.

In [1]:
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

toff = QuantumCircuit(3)
toff.ccx(2, 1, 0)
print("controls = q2,q1   target = q0")
print(toff.draw())

controls = q2,q1   target = q0
     ┌───┐
q_0: ┤ X ├
     └─┬─┘
q_1: ──■──
       │  
q_2: ──■──
          


## Exhaustive truth table

In [2]:
print("c1 c0 t | out")
for c1 in (0, 1):
    for c0 in (0, 1):
        for t in (0, 1):
            qc = QuantumCircuit(3)
            if t:
                qc.x(0)
            if c0:
                qc.x(1)
            if c1:
                qc.x(2)
            qc.ccx(2, 1, 0)
            bits = next(iter(Statevector.from_instruction(qc).to_dict()))
            print(f" {c1}  {c0}  {t} | {bits}")

c1 c0 t | out
 0  0  0 | 000
 0  0  1 | 001
 0  1  0 | 010
 0  1  1 | 011
 1  0  0 | 100
 1  0  1 | 101
 1  1  0 | 111
 1  1  1 | 110


## Reversible AND

In [3]:
print("c1 AND c0  stored on the target")
for c1, c0 in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    qc = QuantumCircuit(3)
    if c0:
        qc.x(1)
    if c1:
        qc.x(2)
    qc.ccx(2, 1, 0)
    bits = next(iter(Statevector.from_instruction(qc).to_dict()))
    print(f"  {c1} AND {c0} = {bits[-1]}   |{bits}>")

c1 AND c0  stored on the target
  0 AND 0 = 0   |000>
  0 AND 1 = 0   |010>
  1 AND 0 = 0   |100>
  1 AND 1 = 1   |111>


## Controls in superposition

Put both controls through $H$. The target should come out $|1\rangle$
only in the $|111\rangle$ outcome — still an AND, now of random bits.

In [4]:
qc = QuantumCircuit(3, 3)
qc.h(1)
qc.h(2)
qc.ccx(2, 1, 0)
qc.measure([0, 1, 2], [0, 1, 2])
sim = AerSimulator()
print(qc.draw())
print(sim.run(transpile(qc, sim), shots=2000).result().get_counts())

          ┌───┐┌─┐      
q_0: ─────┤ X ├┤M├──────
     ┌───┐└─┬─┘└╥┘┌─┐   
q_1: ┤ H ├──■───╫─┤M├───
     ├───┤  │   ║ └╥┘┌─┐
q_2: ┤ H ├──■───╫──╫─┤M├
     └───┘      ║  ║ └╥┘
c: 3/═══════════╩══╩══╩═
                0  1  2 


{'111': 484, '100': 508, '000': 499, '010': 509}
